# Diabetes Evaluation

This notebook uses the same evaluation structure as the other PADME datasets. It compares predictive performance and **model fit time** across Baseline, Random, K-Center and Graph Cut. Fit time includes only `model.fit(...)`; CSV loading, preprocessing and prediction are excluded. Retained-mode times are reported as multipliers relative to the baseline trained in the same notebook.

## 1. Imports

In [ ]:
import json
import warnings
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone

warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Dataset and experiment configuration

In [ ]:
RANDOM_STATE = 1337
DATASET_KEY = "diabetes"
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "padme" / "src" / "main" / "resources" / "data"
OUT_DIR = DATA_DIR / "output" / "diabetes"
TRAIN_PATH = DATA_DIR / "input" / "diabetes_train.csv"
TEST_PATH = OUT_DIR / "diabetes_test.csv"
TARGET_CANDIDATES = ["label"]
TARGET_DTYPE = int

BASELINE_MODE = "baseline"
ALL_MODES = ["baseline", "random", "k_center", "graph_cut"]
MODE_LABELS = {
    "baseline": "Baseline",
    "random": "Random",
    "k_center": "K-Center",
    "graph_cut": "Graph Cut",
}
MODE_STYLE = {
    "baseline": {"color": "black", "marker": None},
    "random": {"color": "tab:orange", "marker": "s"},
    "k_center": {"color": "tab:blue", "marker": "v"},
    "graph_cut": {"color": "tab:red", "marker": "^"},
}

NODES = 5
BASELINE_NODE = 0
TRAINING_REPEATS = 1
DROP_COLS = ["__id"]

RESULTS_DIR = PROJECT_ROOT / "analysis" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.labelcolor": "black",
    "xtick.color": "#444444",
    "ytick.color": "#444444",
    "legend.facecolor": "white",
    "legend.edgecolor": "#cccccc",
})

## 3. Data loading and available runs

In [ ]:
def ratio_to_int(ratio):
    return int(round(float(ratio) * 100))


def load_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Not found: {path}")
    return pd.read_csv(path)


def resolve_target_column(df):
    for candidate in TARGET_CANDIDATES:
        if candidate in df.columns:
            return candidate
    raise ValueError(
        f"None of the target columns {TARGET_CANDIDATES} exists. "
        f"Available columns: {list(df.columns)}"
    )


def split_xy(df, expected_feature_columns=None):
    target_column = resolve_target_column(df)
    y = df[target_column].to_numpy(dtype=TARGET_DTYPE)

    columns_to_drop = set(DROP_COLS) | set(TARGET_CANDIDATES)
    X = df.drop(columns=[c for c in columns_to_drop if c in df.columns], errors="ignore")

    non_numeric = list(X.select_dtypes(exclude=[np.number]).columns)
    if non_numeric:
        raise ValueError(
            "The ML notebooks expect numeric/encoded features. "
            f"Non-numeric columns found: {non_numeric}"
        )

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if expected_feature_columns is not None:
        missing = [c for c in expected_feature_columns if c not in X.columns]
        if missing:
            raise ValueError(f"Missing expected feature columns: {missing}")
        X = X.reindex(columns=expected_feature_columns)

    return X, y, target_column


def discover_mode_ratios(output_root, mode):
    mode_dir = Path(output_root) / mode
    ratios = []
    if not mode_dir.exists():
        return ratios

    for child in mode_dir.iterdir():
        if not child.is_dir():
            continue
        try:
            ratios.append(int(child.name) / 100.0)
        except ValueError:
            continue

    return sorted(set(ratios))


def node_file(mode, ratio, node_idx):
    if mode == BASELINE_MODE:
        return OUT_DIR / BASELINE_MODE / f"{BASELINE_MODE}_node{node_idx}.csv"
    return OUT_DIR / mode / str(ratio_to_int(ratio)) / f"{mode}_node{node_idx}.csv"


def load_node_dataset(mode, ratio, node_idx, expected_feature_columns):
    path = node_file(mode, ratio, node_idx)
    df = load_csv(path)
    X, y, target_column = split_xy(df, expected_feature_columns)
    return X, y, path, target_column


train_reference_df = load_csv(TRAIN_PATH)
test_df = load_csv(TEST_PATH)
X_train_reference, y_train_reference, train_target_column = split_xy(train_reference_df)
FEATURE_COLUMNS = list(X_train_reference.columns)
X_test, y_test, test_target_column = split_xy(test_df, FEATURE_COLUMNS)

AVAILABLE_RATIOS_BY_MODE = {
    mode: discover_mode_ratios(OUT_DIR, mode)
    for mode in ALL_MODES
    if mode != BASELINE_MODE and (OUT_DIR / mode).exists()
}
SWEEP_MODES = [mode for mode in ALL_MODES if mode != BASELINE_MODE and mode in AVAILABLE_RATIOS_BY_MODE]

print("Reference train shape:", X_train_reference.shape)
print("Test shape:", X_test.shape)
print("Resolved train/test targets:", train_target_column, "/", test_target_column)
print("Available ratios by mode:", AVAILABLE_RATIOS_BY_MODE)

## 4. Storage-system metrics

In [ ]:
def load_metrics_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def collect_system_metrics(output_root):
    rows = []

    baseline_metrics = load_metrics_json(Path(output_root) / BASELINE_MODE / "metrics.json")
    if baseline_metrics is not None:
        rows.append({
            "mode": BASELINE_MODE,
            "keep_ratio": 1.0,
            "total_bytes_sent": baseline_metrics.get("totalBytesSent", np.nan),
            "simulation_time_seconds": baseline_metrics.get("simulationTimeSeconds", np.nan),
        })

    for mode in SWEEP_MODES:
        for ratio in AVAILABLE_RATIOS_BY_MODE.get(mode, []):
            metrics = load_metrics_json(
                Path(output_root) / mode / str(ratio_to_int(ratio)) / "metrics.json"
            )
            if metrics is None:
                continue
            rows.append({
                "mode": mode,
                "keep_ratio": float(metrics.get("keepRatio", ratio)),
                "total_bytes_sent": metrics.get("totalBytesSent", np.nan),
                "simulation_time_seconds": metrics.get("simulationTimeSeconds", np.nan),
            })

    metrics_df = pd.DataFrame(rows)
    if metrics_df.empty:
        return metrics_df

    baseline_row = metrics_df[metrics_df["mode"] == BASELINE_MODE]
    if not baseline_row.empty:
        baseline_bytes = float(baseline_row.iloc[0]["total_bytes_sent"])
        baseline_sim_time = float(baseline_row.iloc[0]["simulation_time_seconds"])
        metrics_df["payload_multiplier_vs_baseline"] = metrics_df["total_bytes_sent"] / baseline_bytes
        metrics_df["payload_reduction_pct"] = 100.0 * (1.0 - metrics_df["payload_multiplier_vs_baseline"])
        metrics_df["simulation_time_multiplier_vs_baseline"] = (
            metrics_df["simulation_time_seconds"] / baseline_sim_time
        )

    return metrics_df.sort_values(["mode", "keep_ratio"]).reset_index(drop=True)


system_metrics = collect_system_metrics(OUT_DIR)
display(system_metrics)

if not system_metrics.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    baseline = system_metrics[system_metrics["mode"] == BASELINE_MODE]
    if not baseline.empty:
        axes[0].axhline(
            baseline.iloc[0]["total_bytes_sent"],
            linestyle="--",
            linewidth=2,
            color=MODE_STYLE[BASELINE_MODE]["color"],
            label=MODE_LABELS[BASELINE_MODE],
        )
        axes[1].axhline(1.0, linestyle="--", linewidth=2, color="black", label="Baseline")

    for mode in SWEEP_MODES:
        subset = system_metrics[system_metrics["mode"] == mode].sort_values("keep_ratio")
        if subset.empty:
            continue
        axes[0].plot(
            subset["keep_ratio"],
            subset["total_bytes_sent"],
            marker=MODE_STYLE[mode]["marker"],
            linewidth=2,
            color=MODE_STYLE[mode]["color"],
            label=MODE_LABELS[mode],
        )
        axes[1].plot(
            subset["keep_ratio"],
            subset["simulation_time_multiplier_vs_baseline"],
            marker=MODE_STYLE[mode]["marker"],
            linewidth=2,
            color=MODE_STYLE[mode]["color"],
            label=MODE_LABELS[mode],
        )

    axes[0].set_xlabel("Keep ratio")
    axes[0].set_ylabel("Total bytes sent")
    axes[0].set_title("Replication payload")
    axes[1].set_xlabel("Keep ratio")
    axes[1].set_ylabel("Simulation time multiplier vs baseline")
    axes[1].set_title("Storage simulation time")
    axes[0].legend()
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 5. Retained dataset inventory

In [ ]:
def collect_dataset_inventory():
    rows = []

    baseline_path = node_file(BASELINE_MODE, None, BASELINE_NODE)
    if baseline_path.exists():
        baseline_df = load_csv(baseline_path)
        _, baseline_y, baseline_target = split_xy(baseline_df, FEATURE_COLUMNS)
        rows.append({
            "mode": BASELINE_MODE,
            "keep_ratio": 1.0,
            "node": BASELINE_NODE,
            "rows": len(baseline_y),
            "distinct_targets": int(pd.Series(baseline_y).nunique()),
            "target_column": baseline_target,
            "file": str(baseline_path),
        })

    for mode in SWEEP_MODES:
        for ratio in AVAILABLE_RATIOS_BY_MODE.get(mode, []):
            for node_idx in range(NODES):
                path = node_file(mode, ratio, node_idx)
                if not path.exists():
                    continue
                df = load_csv(path)
                _, y, target_column = split_xy(df, FEATURE_COLUMNS)
                rows.append({
                    "mode": mode,
                    "keep_ratio": ratio,
                    "node": node_idx,
                    "rows": len(y),
                    "distinct_targets": int(pd.Series(y).nunique()),
                    "target_column": target_column,
                    "file": str(path),
                })

    return pd.DataFrame(rows).sort_values(["mode", "keep_ratio", "node"]).reset_index(drop=True)


dataset_inventory = collect_dataset_inventory()
display(dataset_inventory)

## 6. ML model and task-specific metrics

In [ ]:
MODELS = {
    "LogReg": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="saga",
            max_iter=5000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])
}

METRIC_COLUMNS = ["pr_auc", "tp", "fp", "tn", "fn"]
PRIMARY_METRIC = "pr_auc"
PRIMARY_METRIC_LABEL = "PR-AUC"
PRIMARY_METRIC_HIGHER_IS_BETTER = True

def can_train(y_train):
    return len(np.unique(y_train)) >= 2

def score_fitted_model(model, X_eval, y_eval):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predictions = model.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, predictions, labels=[0, 1]).ravel()
    return {
        "pr_auc": float(average_precision_score(y_eval, probabilities)),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

## 7. Timed ML training and evaluation

In [ ]:
def benchmark_model(estimator, X_train, y_train, X_eval, y_eval):
    if not can_train(y_train):
        result = {metric: np.nan for metric in METRIC_COLUMNS}
        result.update({"trained": False, "fit_seconds": np.nan})
        return result

    fit_times = []
    fitted_model = None

    for _ in range(TRAINING_REPEATS):
        fitted_model = clone(estimator)
        started = perf_counter()
        fitted_model.fit(X_train, y_train)
        fit_times.append(perf_counter() - started)

    result = score_fitted_model(fitted_model, X_eval, y_eval)
    result.update({
        "trained": True,
        "fit_seconds": float(np.median(fit_times)),
    })
    return result


def run_all_training_evaluations():
    rows = []

    for model_name, estimator in MODELS.items():
        X_baseline, y_baseline, baseline_path, _ = load_node_dataset(
            BASELINE_MODE, None, BASELINE_NODE, FEATURE_COLUMNS
        )
        baseline_result = benchmark_model(estimator, X_baseline, y_baseline, X_test, y_test)
        rows.append({
            "model": model_name,
            "mode": BASELINE_MODE,
            "keep_ratio": 1.0,
            "node": BASELINE_NODE,
            "train_rows": len(y_baseline),
            "file": str(baseline_path),
            **baseline_result,
        })

        for mode in SWEEP_MODES:
            for ratio in AVAILABLE_RATIOS_BY_MODE.get(mode, []):
                for node_idx in range(NODES):
                    path = node_file(mode, ratio, node_idx)
                    if not path.exists():
                        continue
                    X_node, y_node, path, _ = load_node_dataset(
                        mode, ratio, node_idx, FEATURE_COLUMNS
                    )
                    node_result = benchmark_model(estimator, X_node, y_node, X_test, y_test)
                    rows.append({
                        "model": model_name,
                        "mode": mode,
                        "keep_ratio": ratio,
                        "node": node_idx,
                        "train_rows": len(y_node),
                        "file": str(path),
                        **node_result,
                    })

    return pd.DataFrame(rows)


raw_results = run_all_training_evaluations()
display(raw_results.sort_values(["model", "mode", "keep_ratio", "node"]))

## 8. Aggregated results and baseline-relative measurements

In [ ]:
def sample_std(values):
    values = pd.Series(values).dropna().astype(float)
    if len(values) <= 1:
        return 0.0 if len(values) == 1 else np.nan
    return float(values.std(ddof=1))


def aggregate_training_results(raw_df):
    rows = []

    for model_name in raw_df["model"].drop_duplicates():
        model_rows = raw_df[raw_df["model"] == model_name]
        baseline_rows = model_rows[
            (model_rows["mode"] == BASELINE_MODE) & model_rows["trained"]
        ]
        if baseline_rows.empty:
            raise RuntimeError(f"No valid baseline result for model {model_name}")

        baseline = baseline_rows.iloc[0]
        baseline_fit = float(baseline["fit_seconds"])
        baseline_metric = float(baseline[PRIMARY_METRIC])

        baseline_summary = {
            "model": model_name,
            "mode": BASELINE_MODE,
            "keep_ratio": 1.0,
            "valid_nodes": 1,
            "train_rows_mean": float(baseline["train_rows"]),
            "train_rows_std": 0.0,
            "fit_seconds_mean": baseline_fit,
            "fit_seconds_std": 0.0,
            "fit_time_multiplier_vs_baseline": 1.0,
            "fit_time_multiplier_std": 0.0,
            "training_time_reduction_pct": 0.0,
            "primary_metric_improvement_pct_vs_baseline": 0.0,
        }
        for metric in METRIC_COLUMNS:
            baseline_summary[f"{metric}_mean"] = float(baseline[metric])
            baseline_summary[f"{metric}_std"] = 0.0
        rows.append(baseline_summary)

        sweep_rows = model_rows[
            (model_rows["mode"] != BASELINE_MODE) & model_rows["trained"]
        ]
        for (mode, ratio), group in sweep_rows.groupby(["mode", "keep_ratio"], sort=True):
            fit_mean = float(group["fit_seconds"].mean())
            metric_mean = float(group[PRIMARY_METRIC].mean())

            if PRIMARY_METRIC_HIGHER_IS_BETTER:
                improvement_pct = 100.0 * (metric_mean - baseline_metric) / abs(baseline_metric)
            else:
                improvement_pct = 100.0 * (baseline_metric - metric_mean) / abs(baseline_metric)

            summary_row = {
                "model": model_name,
                "mode": mode,
                "keep_ratio": float(ratio),
                "valid_nodes": int(len(group)),
                "train_rows_mean": float(group["train_rows"].mean()),
                "train_rows_std": sample_std(group["train_rows"]),
                "fit_seconds_mean": fit_mean,
                "fit_seconds_std": sample_std(group["fit_seconds"]),
                "fit_time_multiplier_vs_baseline": fit_mean / baseline_fit,
                "fit_time_multiplier_std": sample_std(group["fit_seconds"]) / baseline_fit,
                "training_time_reduction_pct": 100.0 * (1.0 - fit_mean / baseline_fit),
                "primary_metric_improvement_pct_vs_baseline": improvement_pct,
            }
            for metric in METRIC_COLUMNS:
                summary_row[f"{metric}_mean"] = float(group[metric].mean())
                summary_row[f"{metric}_std"] = sample_std(group[metric])
            rows.append(summary_row)

    return pd.DataFrame(rows).sort_values(["model", "mode", "keep_ratio"]).reset_index(drop=True)


summary_results = aggregate_training_results(raw_results)

raw_path = RESULTS_DIR / f"{DATASET_KEY}_ml_training_raw.csv"
summary_path = RESULTS_DIR / f"{DATASET_KEY}_ml_training_summary.csv"
raw_results.to_csv(raw_path, index=False)
summary_results.to_csv(summary_path, index=False)

print("Raw results:", raw_path)
print("Summary results:", summary_path)
display(summary_results)

## 9. Predictive performance versus baseline

In [ ]:
for model_name in summary_results["model"].drop_duplicates():
    model_summary = summary_results[summary_results["model"] == model_name]
    baseline = model_summary[model_summary["mode"] == BASELINE_MODE].iloc[0]

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.axhline(
        baseline[f"{PRIMARY_METRIC}_mean"],
        linestyle="--",
        linewidth=2,
        color=MODE_STYLE[BASELINE_MODE]["color"],
        label=MODE_LABELS[BASELINE_MODE],
    )

    for mode in SWEEP_MODES:
        subset = model_summary[model_summary["mode"] == mode].sort_values("keep_ratio")
        if subset.empty:
            continue
        ax.errorbar(
            subset["keep_ratio"],
            subset[f"{PRIMARY_METRIC}_mean"],
            yerr=subset[f"{PRIMARY_METRIC}_std"],
            marker=MODE_STYLE[mode]["marker"],
            linewidth=2,
            capsize=3,
            color=MODE_STYLE[mode]["color"],
            label=MODE_LABELS[mode],
        )

    ax.set_xlabel("Keep ratio")
    ax.set_ylabel(PRIMARY_METRIC_LABEL)
    ax.set_title(f"{model_name}: predictive performance")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 10. ML training time versus baseline

In [ ]:
for model_name in summary_results["model"].drop_duplicates():
    model_summary = summary_results[summary_results["model"] == model_name]

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.axhline(1.0, linestyle="--", linewidth=2, color="black", label="Baseline")

    for mode in SWEEP_MODES:
        subset = model_summary[model_summary["mode"] == mode].sort_values("keep_ratio")
        if subset.empty:
            continue
        ax.errorbar(
            subset["keep_ratio"],
            subset["fit_time_multiplier_vs_baseline"],
            yerr=subset["fit_time_multiplier_std"],
            marker=MODE_STYLE[mode]["marker"],
            linewidth=2,
            capsize=3,
            color=MODE_STYLE[mode]["color"],
            label=MODE_LABELS[mode],
        )

    ax.set_xlabel("Keep ratio")
    ax.set_ylabel("ML fit-time multiplier vs baseline")
    ax.set_title(f"{model_name}: model training time")
    ax.legend()
    plt.tight_layout()
    plt.show()

training_time_table = summary_results[[
    "model",
    "mode",
    "keep_ratio",
    "valid_nodes",
    "train_rows_mean",
    "fit_seconds_mean",
    "fit_seconds_std",
    "fit_time_multiplier_vs_baseline",
    "training_time_reduction_pct",
    "primary_metric_improvement_pct_vs_baseline",
]].copy()

display(training_time_table)